# Modelo Final — Issues #13 e #17

Lê exclusivamente artefatos congelados de `reports/modelo-final/`.  
Não treina modelos. Sem outputs.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image

BASE = Path("../reports/modelo-final")

## 1. Validação cruzada — todos os candidatos

In [ ]:
cv = pd.read_csv(BASE / "cv_candidatos.csv")

no_hist = cv[~cv.use_history][["candidate", "ap_mean", "ap_std",
                                "roc_auc_mean", "precision_at_20_mean",
                                "recall_at_20_mean"]].copy()
no_hist.columns = ["Candidato", "AP", "AP ± dp", "ROC-AUC", "Prec@20%", "Rec@20%"]
no_hist

In [ ]:
with_hist = cv[cv.use_history][["candidate", "ap_mean", "ap_std",
                                 "roc_auc_mean", "precision_at_20_mean",
                                 "recall_at_20_mean"]].copy()
with_hist.columns = ["Candidato", "AP", "AP ± dp", "ROC-AUC", "Prec@20%", "Rec@20%"]
with_hist

## 2. Ablação do histórico

In [ ]:
ablation = pd.read_csv(BASE / "cv_ablacao_historico.csv")
ablation

## 3. Escolha congelada

In [ ]:
frozen_path = BASE / "selecao_congelada.json"
if frozen_path.exists():
    frozen = json.loads(frozen_path.read_text())
    for k, v in frozen.items():
        print(f"{k:25s}: {v}")
else:
    print("selecao_congelada.json ainda não existe. Execute --fase freeze.")

## 4. Resultado no teste reservado (avaliação única)

In [ ]:
teste_path = BASE / "teste_final.csv"
if teste_path.exists():
    teste = pd.read_csv(teste_path)
    display(teste[["candidate", "ap", "roc_auc", "precision_at_20",
                   "recall_at_20", "f1_at_threshold"]])
else:
    print("teste_final.csv ainda não existe. Execute --fase final.")

## 5. SHAP — importância das variáveis

In [ ]:
shap_var_path = BASE / "shap_por_variavel.csv"
if shap_var_path.exists():
    shap_var = pd.read_csv(shap_var_path)
    # Agregação: por linha, soma SHAP com sinal das colunas OHE da variável → mean(|soma|)
    display(shap_var)

var_fig = BASE / "figures" / "shap_por_variavel.png"
if var_fig.exists():
    print("Importância por variável original — mean|SHAP agregado por linha|:")
    display(Image(str(var_fig)))

print("\n--- Top 10 pós-OHE (referência por coluna) ---")
shap_top_path = BASE / "shap_top10.csv"
if shap_top_path.exists():
    display(pd.read_csv(shap_top_path))

bar_path = BASE / "figures" / "shap_bar.png"
if bar_path.exists():
    display(Image(str(bar_path)))

bee_path = BASE / "figures" / "shap_beeswarm.png"
if bee_path.exists():
    display(Image(str(bee_path)))

## 6. Análise de erros no teste

In [ ]:
err_path = BASE / "error_analysis.csv"
if err_path.exists():
    err = pd.read_csv(err_path)
    # Mostra falta_rate ao lado de fn_rate para contextualizar a política dos 20%
    cols = ["group", "falta_rate", "fp_rate", "fn_rate", "n_total", "n_faltas"]
    for dim, grp in err.groupby("dimension"):
        print(f"\n--- {dim} ---")
        show_cols = [c for c in cols if c in grp.columns]
        display(grp[show_cols].reset_index(drop=True))
    print(
        "\nNota: taxa de FN alta em grupos com falta_rate < 20,3% (média global) é "
        "consequência da política top-20% aplicada a grupos de menor risco, não falha do modelo."
    )
else:
    print("error_analysis.csv ainda não existe. Execute --fase final.")